# 13. Workflows / Jobs - 順番・リトライ・通知を任せる

ここまでの12本は、**ノートブックを手で上から実行** してきました。
学ぶにはよいのですが、運用はこの形では回りません。

- 毎朝だれかが実行ボタンを押すのか
- 途中で落ちたら、どこからやり直すのか
- 落ちたことに、どうやって気づくのか

これを引き受けるのが **ジョブ (Workflows)** です。
`08` のLDPが「テーブルの作り方」を宣言する仕組みだったのに対し、
ジョブは **「処理の順番と、失敗したときどうするか」** を宣言する仕組みです。

このノートブックで確かめること:

1. タスクの依存関係をどう書くか
2. タスクが失敗すると、後続はどうなるか
3. リトライがどう効くか
4. LDP と、どちらを選ぶか

**前提**: `00_setup` を実行済みであること。`08` を読んでいること。


## 準備


In [ ]:
import time

from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import Run, RunLifeCycleState

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()
w = WorkspaceClient(profile="free")

In [ ]:
CATALOG = "tech_survey"

# resources/jobs/13_workflow.job.yml で付けた名前
JOB_NAME = "13_survey_job"

## 1. タスクの中身

タスクは3つです。ソースは
[`src/jobs/13_workflow/`](../../jobs/13_workflow) にあります。

これまでのノートブックと違い、**ローカルでは実行しません。**
Databricks側でジョブとして動くファイルなので、`.ipynb` ではなく
`# Databricks notebook source` で始まる `.py` 形式にしてあります。

| タスク | やること |
|---|---|
| `ingest` | bronze に2件書く |
| `transform` | bronze を silver にコピーする。**わざと失敗させられる** |
| `publish` | silver を集計して gold に書く |

`transform` だけ、こうなっています。

```python
dbutils.widgets.text("fail_transform", "false")
fail_transform = dbutils.widgets.get("fail_transform")

if fail_transform == "true":
    raise RuntimeError("わざと失敗させている (fail_transform=true)")
```

実行するたびに外から値を渡せる仕組みです。
**コードを書き換えずに、成功させたり失敗させたりできます。**


## 2. ジョブ定義を読む

[`resources/jobs/13_workflow.job.yml`](../../../resources/jobs/13_workflow.job.yml) がジョブの定義です。

```yaml
tasks:
  - task_key: ingest
    notebook_task:
      notebook_path: ../../src/jobs/13_workflow/ingest.py

  - task_key: transform
    depends_on:                       # ingest が成功してから動く
      - task_key: ingest
    max_retries: 1                    # 失敗しても1回だけやり直す
    min_retry_interval_millis: 10000  # やり直すまで10秒あける
    notebook_task:
      notebook_path: ../../src/jobs/13_workflow/transform.py
      base_parameters:
        fail_transform: "{{job.parameters.fail_transform}}"

  - task_key: publish
    depends_on:
      - task_key: transform
```

**`depends_on` が順番のすべてです。**
`08` のLDPでは、テーブル名の参照から順番が自動で決まっていました。
ジョブでは **人が明示的に書きます。** タスクの中身が何であれ関係なく、書いたとおりに並びます。

融通が利く代わりに、**依存の書き漏らしは誰も教えてくれません。**
`transform` が `ingest` の結果を使うのに `depends_on` を書き忘れると、両方が同時に走ります。


## 3. デプロイして起動する

`08` と同じくデプロイが要ります。リポジトリのルートで実行してください。

```sh
databricks bundle deploy
```


In [ ]:
# dev ターゲットでは名前に接頭辞が付くので、後方一致で探す
JOB_ID = next(j.job_id for j in w.jobs.list() if j.settings.name.endswith(JOB_NAME))

# この画面で、タスクの依存関係が図として見える
print(f"{w.config.host}/jobs/{JOB_ID}")

In [ ]:
# 実行が終わったことを表す状態
DONE = (
    RunLifeCycleState.TERMINATED,
    RunLifeCycleState.SKIPPED,
    RunLifeCycleState.INTERNAL_ERROR,
)


def run_job(fail_transform: str = "false") -> Run:
    """
    ジョブを起動し、終わるまで待つ。

    Parameters
    ----------
    fail_transform : str, default "false"
        "true" を渡すと transform タスクがわざと失敗する。

    Returns
    -------
    Run
        終了時点の実行情報。タスクごとの状態は `.tasks` に入っている。

    Notes
    -----
    サーバーレスでもタスクの起動には時間がかかる。
    途中経過が分かるよう、15秒ごとに状態を表示する。
    """
    started = w.jobs.run_now(job_id=JOB_ID, job_parameters={"fail_transform": fail_transform})

    while True:
        current = w.jobs.get_run(started.run_id)
        state = current.state.life_cycle_state
        print(state.value)
        if state in DONE:
            return current
        time.sleep(15)


def show_tasks(run: Run) -> None:
    """
    タスクごとの結果を1行ずつ表示する。

    Parameters
    ----------
    run : Run
        `run_job` が返した実行情報。
    """
    for task in sorted(run.tasks, key=lambda t: t.task_key):
        result = task.state.result_state
        print(
            f"{task.task_key:10} "
            f"試行{task.attempt_number}回目  "
            f"{task.state.life_cycle_state.value:12} "
            f"{result.value if result else '-'}"
        )

## 4. わざと失敗させる

`fail_transform="true"` で起動します。`transform` が必ず失敗します。

このとき、**3つのタスクがそれぞれどうなるか** を予想してください。

- `ingest` は？
- `transform` は何回動く？
- `publish` は？


In [ ]:
failed_run = run_job(fail_transform="true")

In [ ]:
show_tasks(failed_run)

3つとも違う結果になったはずです。

- **`ingest`** … 成功している。`transform` が落ちても、**終わった仕事は取り消されません**
- **`transform`** … 失敗。`max_retries: 1` を付けたので **2回試して** から諦めています
- **`publish`** … `UPSTREAM_FAILED`。**自分は1度も動いていません**

`publish` の状態が `FAILED` ではなく `UPSTREAM_FAILED` なのが大事なところです。
「自分が壊れた」のか「前の工程が届かなかった」のかが区別されています。
落ちたときに **どこを直せばよいか** が、この区別で分かります。

ここで注意が要るのが `ingest` です。
**成功したぶんは元に戻りません。** ジョブ全体が失敗しても、bronze には既に書かれています。

だから `04`〜`06` でやった **冪等性** が効いてきます。
やり直しても二重にならない書き方をしておけば、失敗した回の続きから流し直せます。
そうでないと、手でデータを消してから再実行する羽目になります。
**ジョブのリトライは、冪等な処理とセットで初めて意味を持ちます。**


## 5. 直して流す

今度は `fail_transform="false"` (既定) で起動します。
コードもジョブ定義も変えていません。**渡す値だけが違います。**


In [ ]:
ok_run = run_job()
show_tasks(ok_run)

In [ ]:
# 3つのタスクが書いたテーブルを確認する
for t in ("bronze.job_orders", "silver.job_orders", "gold.job_summary"):
    print(t, spark.table(f"{CATALOG}.{t}").count(), "件")

display(spark.table(f"{CATALOG}.gold.job_summary"))

全部 `SUCCESS` になり、gold まで到達しました。

`12` の最後に「`addNewColumns` は1回落ちるが、再実行で通る」と書きました。
その再実行を引き受けるのが、ここで見た `max_retries` です。

ただし **回数は必ず決めます。** 無限にやり直すと、
本当に直らない失敗 (壊れたファイル、権限不足) が延々と再試行され、
**失敗したことに気づけなくなります。**

気づくための仕組みは、ジョブ定義に書けます。

```yaml
email_notifications:
  on_failure:
    - someone@example.com
```

`on_start` / `on_success` / `on_failure` を指定できます。
実務で付ける価値があるのは **`on_failure` だけ** です。
成功のたびに通知すると、そのうち誰も読まなくなります。


## 6. LDP とジョブ、どちらを選ぶか

`08` と見比べます。どちらも「複数の処理をまとめて動かす」ものですが、宣言している対象が違います。

| | LDP (`08`) | ジョブ (`13`) |
|---|---|---|
| 宣言するもの | テーブルの中身 | 処理の順番 |
| 実行順 | 依存から自動で決まる | `depends_on` に人が書く |
| 扱える処理 | テーブルを作ること | 何でも (ノートブック、SQL、他のジョブ、**パイプラインの起動**) |
| 失敗したとき | パイプライン単位で止まる | タスク単位で止まり、後続はスキップ |
| リトライ | エンジンが面倒を見る | `max_retries` に自分で書く |

**排他ではありません。** 実際によくある構成はこうです。

1. ジョブのタスク1 … 外部システムからファイルを取ってくる
2. ジョブのタスク2 … **LDPパイプラインを起動する** (`pipeline_task`)
3. ジョブのタスク3 … 完了後に通知を出す

テーブルを作る部分はLDPに任せ、**その前後の段取りをジョブが持つ** という分担です。
「順番を自動で決めてほしい」のか「決まった順番どおりに動かしたい」のかで選ぶと迷いません。


## 考えてみる

- `ingest` が成功した後に `transform` が失敗しました。もう一度ジョブを流すと `ingest` はどうなりますか
- `max_retries` を10にすると、何が困るでしょうか
- スケジュール実行にしたとき、前の実行がまだ終わっていなかったらどうなりますか


### 答え

**Q1. 再実行したときの `ingest`**

**もう一度、最初から実行されます。** 「成功したタスクを飛ばす」ということはしません。

なので `ingest` は2回動くことになります。
中身が `INSERT` の追記だったら、データが二重になります。
このノートブックの `ingest` が `CREATE OR REPLACE TABLE` なのは、
**何度実行しても同じ結果になるようにする** ためです。

なお、失敗したタスクから再開する機能 (`Repair run`) はあります。
ただしそれは復旧の手段であって、**冪等でなくてよい理由にはなりません。**

**Q2. `max_retries` を10にすると**

**直らない失敗に気づくのが遅れます。**

権限不足や壊れたファイルが原因なら、10回やっても結果は同じです。
そのぶん時間とコストを使い、通知も最後まで飛びません。

リトライが効くのは **「たまたま失敗した」場合だけ** です。
一時的な接続エラー、`12` で見たスキーマ更新、他の処理との衝突 (`11` の `ConcurrentWriteException`) など。
こうした失敗は1〜2回で通るので、回数を増やす意味は小さくなります。

**Q3. 前の実行が終わっていないとき**

既定では **同時に走りません。** 新しい実行はスキップされます。

`max_concurrent_runs` で同時実行数を増やせますが、
**増やす前に「同時に走って平気か」を確認する** 必要があります。
同じテーブルに書く処理が2つ同時に走れば、`11` で見た書き込みの衝突が起きます。

処理が1時間かかるのに30分おきに起動する、という設定になっていないか。
スケジュールを決めるときは、**処理時間より短い間隔にしない** のが基本です。


## 後片付け

このノートブックで作ったものを消したいときだけ、コメントを外して実行します。
ジョブ自体は `databricks bundle destroy` で消せます。


In [ ]:
# for t in ("bronze.job_orders", "silver.job_orders", "gold.job_summary"):
#     spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{t}")